<a href="https://colab.research.google.com/github/Pradeep-Kumar-Panga/DecodingComplexities/blob/main/NaiveBayesClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive Bayes Classifier

## What is a Naive Bayes Classifier?

**Naive Bayes** is a probabilistic classification algorithm based on **Bayes' Theorem**. It predicts the class of an observation by calculating the probability that it belongs to each possible class, then choosing the class with the highest probability.

**Key Formula** (Bayes' Theorem):

$$P(\text{Class}|\text{Features}) = \frac{P(\text{Features}|\text{Class}) \cdot P(\text{Class})}{P(\text{Features})}$$

Where:
- $P(\text{Class}|\text{Features})$ = Posterior probability (what we want to find)
- $P(\text{Features}|\text{Class})$ = Likelihood (probability of features given the class)
- $P(\text{Class})$ = Prior probability (how common the class is)
- $P(\text{Features})$ = Evidence (probability of the features occurring)

## The "Naive" Assumption

The classifier is called "naive" because it assumes that **all features are conditionally independent** given the class. This means:

$$P(\text{Features}|\text{Class}) = P(f_1|\text{Class}) \times P(f_2|\text{Class}) \times ... \times P(f_n|\text{Class})$$

In reality, features are often correlated, but this simplifying assumption:
- Makes computation much faster
- Requires less training data
- Still works surprisingly well in practice

## Example Application: Spam Detection

Let's apply Naive Bayes to classify emails as spam or not spam based on the words they contain.

## The Math

**Bayes' Theorem for Classification**:

$$P(\text{Spam}|\text{Words}) = \frac{P(\text{Words}|\text{Spam}) \cdot P(\text{Spam})}{P(\text{Words})}$$

**Naive Assumption**: Words are independent, so:

$$P(\text{Words}|\text{Spam}) = P(w_1|\text{Spam}) \times P(w_2|\text{Spam}) \times ... \times P(w_n|\text{Spam})$$

**Classification Rule**: Choose class with higher probability. Ignore denominator (same for both).

$$\text{Prediction} = \arg\max_{\text{class}} P(\text{class}) \prod_{i} P(w_i|\text{class})$$

## Simple Example:
4 sample emails with 2 spam and 2 not spam emails.

In [1]:
import numpy as np

# Super simple dataset - just 4 emails, already cleaned (lowercase, no punctuation)
simple_emails = [
    "free money win",           # Spam
    "meeting tomorrow project", # Not spam
    "free click win prize",     # Spam
    "report meeting deadline"   # Not spam
]

simple_labels = [1, 0, 1, 0]  # 1=Spam, 0=Not spam

print("Our sample dataset:")
for i, (email, label) in enumerate(zip(simple_emails, simple_labels)):
    print(f"{i+1}. [{('Not spam', 'Spam')[label]}] {email}")

Our sample dataset:
1. [Spam] free money win
2. [Not spam] meeting tomorrow project
3. [Spam] free click win prize
4. [Not spam] report meeting deadline


### Step 1: Prior Probability

Calculate the prior probability of Spam and Not spam emails from the given dataset.

In [2]:
n_spam = sum(simple_labels)
n_not_spam = len(simple_labels) - n_spam

P_spam = n_spam / len(simple_labels)
P_notspam = n_not_spam / len(simple_labels)

print(f"P(Spam) = {P_spam}, P(Not spam) = {P_notspam}")

P(Spam) = 0.5, P(Not spam) = 0.5


### Step 2: Word Counts

Calculate the count of each word in the input data set from both Spam and Not spam emails.
These counts are used to calculate the probability of being spam or not spam given a new email.

In [3]:
# Count words in spam and not spam emails
word_count_spam = {}
word_count_notspam = {}

for email, label in zip(simple_emails, simple_labels):
    words = email.split()
    for word in words:
        if label == 1:  # Spam
            word_count_spam[word] = word_count_spam.get(word, 0) + 1
        else:  # Not spam
            word_count_notspam[word] = word_count_notspam.get(word, 0) + 1

print("Word counts in SPAM emails:")
for word, count in sorted(word_count_spam.items()):
    print(f"  {word}: {count}")

print("\nWord counts in NOT SPAM emails:")
for word, count in sorted(word_count_notspam.items()):
    print(f"  {word}: {count}")

Word counts in SPAM emails:
  click: 1
  free: 2
  money: 1
  prize: 1
  win: 2

Word counts in NOT SPAM emails:
  deadline: 1
  meeting: 2
  project: 1
  report: 1
  tomorrow: 1


### The Zero-Probability Problem

What happens if we encounter a word that never appeared in training? Let's see:

In [4]:
# Calculate probabilities WITHOUT smoothing
all_words = set(' '.join(simple_emails).split())
total_spam = sum(word_count_spam.values())
total_notspam = sum(word_count_notspam.values())

# Let's test with a new word that wasn't in training
new_word = "urgent"
print(f"Testing with new word: '{new_word}'")
print(f"Count in spam: {word_count_spam.get(new_word, 0)}")
print(f"Count in not spam: {word_count_notspam.get(new_word, 0)}")

# Calculate probability WITHOUT smoothing
prob_urgent_spam = word_count_spam.get(new_word, 0) / total_spam
prob_urgent_notspam = word_count_notspam.get(new_word, 0) / total_notspam

print(f"\nP({new_word}|spam) = {word_count_spam.get(new_word, 0)}/{total_spam} = {prob_urgent_spam}")
print(f"P({new_word}|not spam) = {word_count_notspam.get(new_word, 0)}/{total_notspam} = {prob_urgent_notspam}")

# Now try to classify: "urgent free money"
print(f"\n⚠️  Problem: Classifying 'urgent free money'")
print(f"P(spam) × P(urgent|spam) × P(free|spam) × P(money|spam)")
print(f"= 0.5 × {prob_urgent_spam} × ... = 0")
print("\n❌ The entire probability becomes ZERO! Classification breaks!")

Testing with new word: 'urgent'
Count in spam: 0
Count in not spam: 0

P(urgent|spam) = 0/7 = 0.0
P(urgent|not spam) = 0/6 = 0.0

⚠️  Problem: Classifying 'urgent free money'
P(spam) × P(urgent|spam) × P(free|spam) × P(money|spam)
= 0.5 × 0.0 × ... = 0

❌ The entire probability becomes ZERO! Classification breaks!


### Step 3: Solution - Laplace Smoothing (add 1)

To avoid zero probabilities, we add 1 to all counts:

In [5]:
vocab_size = len(all_words)

word_prob_spam = {}
word_prob_notspam = {}

for word in all_words:
    # Add 1 to numerator, vocab_size to denominator
    word_prob_spam[word] = (word_count_spam.get(word, 0) + 1) / (total_spam + vocab_size)
    word_prob_notspam[word] = (word_count_notspam.get(word, 0) + 1) / (total_notspam + vocab_size)

print(f"{'Word':<12} {'P(w|spam)':<12} {'P(w|not spam)':<12}")
for word in sorted(all_words):
    print(f"{word:<12} {word_prob_spam[word]:.4f}      {word_prob_notspam[word]:.4f}")

# Now the unseen word gets a small probability instead of zero
prob_urgent_spam_smoothed = 1 / (total_spam + vocab_size)
print(f"\n✅ With smoothing: P(urgent|spam) = {prob_urgent_spam_smoothed:.4f} (not zero!)")

Word         P(w|spam)    P(w|not spam)
click        0.1176      0.0625
deadline     0.0588      0.1250
free         0.1765      0.0625
meeting      0.0588      0.1875
money        0.1176      0.0625
prize        0.1176      0.0625
project      0.0588      0.1250
report       0.0588      0.1250
tomorrow     0.0588      0.1250
win          0.1765      0.0625

✅ With smoothing: P(urgent|spam) = 0.0588 (not zero!)


### Step 4: Classify "free money click"

In [6]:
test_email = "free money click"
test_words = test_email.split()

log_prob_spam = np.log(P_spam)
log_prob_notspam = np.log(P_notspam)

for word in test_words:
    log_prob_spam += np.log(word_prob_spam.get(word, 1/vocab_size))
    log_prob_notspam += np.log(word_prob_notspam.get(word, 1/vocab_size))

print(f"Test: '{test_email}'")
print(f"Log P(spam|email) = {log_prob_spam:.4f}")
print(f"Log P(not spam|email) = {log_prob_notspam:.4f}")
print(f"\nPrediction: {'SPAM' if log_prob_spam > log_prob_notspam else 'NOT SPAM'}")

Test: 'free money click'
Log P(spam|email) = -6.7079
Log P(not spam|email) = -9.0109

Prediction: SPAM


**Key Points**:
- Spam words (free, money) have higher P(word|spam)
- Laplace smoothing prevents zero probabilities for unseen words
- Use log probabilities to avoid underflow
- Ignore denominator (same for both classes)

---

## Full Implementation

### Libraries

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import re

np.random.seed(42)

### Dataset (30 emails)

In [8]:
# Create a dataset of sample emails
emails = [
    # Spam emails (label = 1)
    "Congratulations! You've won a free lottery ticket worth $1000. Click here now!",
    "Get cheap medicines online. Viagra, Cialis available. Order now!",
    "URGENT: Your account needs verification. Click this link immediately!",
    "Make money fast! Work from home and earn $5000 per week!",
    "Free discount on luxury watches! Limited time offer. Buy now!",
    "You have inherited millions from a Nigerian prince. Send bank details!",
    "Winner winner! Claim your prize of $10000 cash. Act now!",
    "Lose weight fast with our miracle pill. Free trial available!",
    "Cheap loans available. Bad credit OK. Apply now for instant approval!",
    "Click here for free iPhone. Limited stock remaining!",
    "Get rich quick with our investment scheme. Guaranteed returns!",
    "Free casino bonus! Play now and win big money!",
    "Enlarge your income with this amazing opportunity! No experience needed!",
    "Claim your free vacation to Bahamas. Limited spots available!",
    "Urgent: Update your password immediately. Click link below.",

    # Not spam emails (label = 0)
    "Hi John, let's meet for coffee tomorrow at 3pm. Looking forward to it!",
    "Team meeting scheduled for Monday at 10am. Please confirm attendance.",
    "Your package has been shipped and will arrive by Friday.",
    "Mom called and wants to know if you're coming for dinner on Sunday.",
    "The project deadline has been extended to next week. Please update your plans.",
    "Thank you for your purchase. Your order number is 12345.",
    "Reminder: Doctor appointment scheduled for Tuesday at 2pm.",
    "Can you review this document and send me your feedback by Thursday?",
    "Happy birthday! Hope you have a wonderful day with family and friends.",
    "The quarterly report is ready for your review. Let me know if you need changes.",
    "Your subscription has been renewed successfully. Thank you!",
    "Meeting notes from yesterday are attached. Please review at your convenience.",
    "Looking forward to our call tomorrow. Talk to you soon!",
    "The software update has been completed successfully on your account.",
    "Your flight confirmation for next Monday. Have a safe trip!",
]

# Labels: 1 for spam, 0 for not spam
labels = [1] * 15 + [0] * 15

# Create DataFrame
df = pd.DataFrame({
    'email': emails,
    'label': labels,
    'class': ['spam' if l == 1 else 'not spam' for l in labels]
})

print(f"Dataset size: {len(df)} emails")
print(f"Spam emails: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
print(f"Not spam emails: {len(labels) - sum(labels)} ({(len(labels) - sum(labels))/len(labels)*100:.1f}%)")
print("\nFirst few emails:")
df.head()

Dataset size: 30 emails
Spam emails: 15 (50.0%)
Not spam emails: 15 (50.0%)

First few emails:


,email,label,class
0,Congratulations! You've won a free lottery tic...,1,spam
1,"Get cheap medicines online. Viagra, Cialis ava...",1,spam
2,URGENT: Your account needs verification. Click...,1,spam
3,Make money fast! Work from home and earn $5000...,1,spam
4,Free discount on luxury watches! Limited time ...,1,spam


### Text Preprocessing

In [9]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

df['tokens'] = df['email'].apply(preprocess_text)
print(f"Example: {df['tokens'].iloc[0][:5]}...")

Example: ['congratulations', 'youve', 'won', 'a', 'free']...


### Train/Test Split (70/30)

In [10]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
print(f"Train: {len(train_df)} emails | Test: {len(test_df)} emails")

Train: 21 emails | Test: 9 emails


### Training

**Step 1: Build Vocabulary**

In [11]:
vocabulary = set()
for tokens in train_df['tokens']:
    vocabulary.update(tokens)
vocab_list = sorted(list(vocabulary))
vocab_size = len(vocab_list)
print(f"Vocabulary: {vocab_size} words")

Vocabulary: 149 words


**Step 2: Priors**

In [12]:
prior_spam = sum(train_df['label']) / len(train_df)
prior_not_spam = 1 - prior_spam
print(f"P(spam) = {prior_spam:.3f}, P(not spam) = {prior_not_spam:.3f}")

P(spam) = 0.524, P(not spam) = 0.476


**Step 3: Word Probabilities (Laplace smoothing)**

In [13]:
word_count_spam = defaultdict(int)
word_count_not_spam = defaultdict(int)

for idx, row in train_df.iterrows():
    for word in row['tokens']:
        if row['label'] == 1:
            word_count_spam[word] += 1
        else:
            word_count_not_spam[word] += 1

total_words_spam = sum(word_count_spam.values())
total_words_not_spam = sum(word_count_not_spam.values())

word_prob_spam = {}
word_prob_not_spam = {}

for word in vocab_list:
    word_prob_spam[word] = (word_count_spam[word] + 1) / (total_words_spam + vocab_size)
    word_prob_not_spam[word] = (word_count_not_spam[word] + 1) / (total_words_not_spam + vocab_size)

print("Sample word probabilities:")
for word in ['free', 'money', 'meeting']:
    if word in vocab_list:
        print(f"{word}: P(spam)={word_prob_spam[word]:.4f}, P(not spam)={word_prob_not_spam[word]:.4f}")

Sample word probabilities:
free: P(spam)=0.0157, P(not spam)=0.0039
money: P(spam)=0.0079, P(not spam)=0.0039
meeting: P(spam)=0.0039, P(not spam)=0.0117


**Step 4: Classification Function**

In [14]:
def classify_email(email_text):
    words = preprocess_text(email_text)
    log_prob_spam = np.log(prior_spam)
    log_prob_not_spam = np.log(prior_not_spam)

    for word in words:
        if word in vocab_list:
            log_prob_spam += np.log(word_prob_spam[word])
            log_prob_not_spam += np.log(word_prob_not_spam[word])

    return (1 if log_prob_spam > log_prob_not_spam else 0), log_prob_spam, log_prob_not_spam

# Test
test_email = "Congratulations! You won free money. Click here now!"
pred, log_spam, log_not_spam = classify_email(test_email)
print(f"Test: {test_email[:40]}...")
print(f"Prediction: {'SPAM' if pred == 1 else 'NOT SPAM'}")

Test: Congratulations! You won free money. Cli...
Prediction: SPAM


### Evaluate on Test Set

In [15]:
test_predictions = [classify_email(email)[0] for email in test_df['email']]
accuracy = accuracy_score(test_df['label'], test_predictions)
cm = confusion_matrix(test_df['label'], test_predictions)

print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"\nConfusion Matrix:")
print(f"         Not spam  Spam")
print(f"Not spam {cm[0,0]:<4} {cm[0,1]:<4}")
print(f"Spam     {cm[1,0]:<4} {cm[1,1]:<4}")

Accuracy: 100.0%

Confusion Matrix:
         Not spam  Spam
Not spam 5    0   
Spam     0    4   


### Compare with Scikit-learn

In [16]:
# Create feature matrix
def create_feature_matrix(df, vocabulary):
    matrix = np.zeros((len(df), len(vocabulary)))
    for i, tokens in enumerate(df['tokens']):
        for word in tokens:
            if word in vocabulary:
                matrix[i, vocab_list.index(word)] += 1
    return matrix

X_train = create_feature_matrix(train_df, vocab_list)
X_test = create_feature_matrix(test_df, vocab_list)

sklearn_nb = MultinomialNB(alpha=1.0)
sklearn_nb.fit(X_train, train_df['label'].values)
sklearn_accuracy = accuracy_score(test_df['label'], sklearn_nb.predict(X_test))

print(f"Scikit-learn: {sklearn_accuracy * 100:.1f}%")
print(f"Our implementation: {accuracy * 100:.1f}%")

Scikit-learn: 100.0%
Our implementation: 100.0%


### Word Importance

In [17]:
word_ratios = {w: np.log(word_prob_spam[w]) - np.log(word_prob_not_spam[w]) for w in vocab_list}
sorted_words = sorted(word_ratios.items(), key=lambda x: x[1])

print("Top 5 SPAM indicators:")
for word, ratio in sorted_words[-5:][::-1]:
    print(f"  {word}: {ratio:.3f}")

print("\nTop 5 NOT SPAM indicators:")
for word, ratio in sorted_words[:5]:
    print(f"  {word}: {ratio:.3f}")

Top 5 SPAM indicators:
  click: 1.621
  now: 1.398
  free: 1.398
  winner: 1.110
  urgent: 1.110

Top 5 NOT SPAM indicators:
  at: -1.598
  been: -1.087
  has: -1.087
  is: -1.087
  meeting: -1.087


## Summary

**How it works**:
1. Train: Calculate P(spam), P(not spam), and P(word|class) for each word
2. Classify: Multiply probabilities (use logs to avoid underflow)
3. Choose class with higher probability

**Why "naive"**: Assumes words are independent (they're not, but it works well anyway)

**Strengths**: Fast, simple, works with small data, handles high dimensions

**Limitations**: Can't learn word relationships, needs smoothing for unseen words